[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_H.ipynb)


# Set H — BMI‑1: micro:bit Only (LEGO–Colab)
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

> Local runtime required for serial access. Provides a FEAR command bridge to micro:bit over USB UART.

---
## Table of Contents
- [H0 Starter](#h0)
- [H1 FEAR activation logic (host)](#h1)
- [H2 micro:bit firmware (flash as main.py)](#h2)
- [Reflection](#reflection)
---


## H0 — Starter (local runtime only) <a id='h0'></a>

In [ ]:

!pip -q install pyserial
import time, sys, serial, serial.tools.list_ports as list_ports
VID_MICROBIT=3368; PID_MICROBIT=516; BAUD=115200
_ser=None

def find_port():
    for p in list_ports.comports():
        if getattr(p,'vid',None)==VID_MICROBIT and getattr(p,'pid',None)==PID_MICROBIT:
            return p.device
    for p in list_ports.comports():
        d=(p.description or '').lower()
        if any(x in d for x in ['micro:bit','mbed','daplink','bbc']): return p.device
    return None

def open_serial():
    global _ser
    if _ser and _ser.is_open: return _ser
    port=find_port()
    if not port: raise RuntimeError('micro:bit not found')
    _ser=serial.Serial(port,BAUD,timeout=0.2); time.sleep(0.2); return _ser

def send_fear():
    s=open_serial(); s.write(b'FEAR
'); s.flush()

print('H0 ready.')


## H1 — FEAR activation logic (host side) <a id='h1'></a>

In [ ]:

import ipywidgets as w
from IPython.display import display

thr=w.FloatSlider(value=0.7,min=0,max=1,step=0.01,description='FEAR thr')
b=w.Button(description='Test FEAR',button_style='danger')

def on_fear(prob,thr=0.7,cd_ms=1500):
    if not hasattr(on_fear,'t0'): on_fear.t0=0
    now=time.time()*1000
    if prob>=thr and now-on_fear.t0>cd_ms:
        send_fear(); on_fear.t0=now; print('FEAR sent',prob)

def _t(_): send_fear()
b.on_click(_t)
display(w.VBox([thr,b]))


## H2 — micro:bit firmware (flash as main.py) <a id='h2'></a>

In [ ]:

from microbit import *
import music, uart
uart.init(baudrate=115200)

def fear():
    display.show(Image.SKULL)
    music.play(['c5:1','b4:1','a4:2'])
    display.clear()

while True:
    if uart.any():
        if uart.readline().strip().upper()==b'FEAR': fear()
    sleep(10)


## Reflection <a id='reflection'></a>
- Expand to multi‑command protocol (e.g., CALM, GO).
- Decide where refractory/debouncing lives (host vs device).
- Bridge ML outputs (probabilities) to reliable triggers.


---

## Practice / Discussion Questions — Set H — Translation to BMI

1) Define an **internal model state** to export to a device and why it’s meaningful.
2) *Design*: Choose a **threshold rule** to convert a scalar state into a trigger; justify.
3) How would you **measure latency** from state crossing to device action?
4) One **communication error mode** and its mitigation.
5) An **encoding** for graded states (levels/PWM) and why it helps.
6) *Predict*: What mismatch occurs if internal events are faster than the effector; fix?
7) Why emphasize **transparent I/O semantics** in educational BMIs?
8) Create a **simple interface contract** and a verification test.
9) Two sources of **timing jitter** to check if actions are inconsistent.
10) How to **log events** to timestamp internal and external actions.
11) One reason to **debounce** or smooth state transitions before triggering.
12) How to document **assumptions** so others can reproduce your mapping.
13) One **ethical/interpretive** caution mapping state → action.
14) *Extend*: Add a second effector representing a different property.
15) Propose a **stress test** and pass/fail criteria.
16) Why a BMI lab is a strong **capstone** for model→behavior.
17) One **accessibility** consideration in signal→action mapping.
18) How to **simulate** device logic before hardware.
19) *Compare*: Direct thresholding vs **hysteresis window**—which and why?
20) How Set H reinforces **mechanistic understanding** rather than replacing it.
